# 🔧 Support Vector Machines
**ISLP Chapter 9 · Data Pattern Recognition for the Rest of Us**

> SVMs find the hyperplane that maximally separates classes. With kernels, they handle nonlinear boundaries. They work well in high dimensions and with imbalanced classes — making them a natural fit for fraud detection.

### Dataset
**Credit Card Fraud Detection**
Synthetic dataset modelled on the properties of the real European credit card fraud dataset (Kaggle). 284,807 transactions, 0.17% fraud rate — a classic imbalanced classification challenge in financial services.

### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)
from sklearn.datasets import make_classification
print("\u2713 Setup complete")

In [ ]:
# Credit Card Fraud — realistic synthetic dataset
# Properties match the real Kaggle dataset:
# - Highly imbalanced (0.17% fraud)
# - PCA-transformed features (V1-V8) + Amount + Time
np.random.seed(42)
n_legit, n_fraud = 10000, 17  # ~0.17% fraud rate

# Legitimate transactions
legit = np.random.multivariate_normal(
    mean=np.zeros(8),
    cov=np.eye(8) + 0.2*np.random.rand(8,8),
    size=n_legit)
legit_amount = np.random.lognormal(4.5, 1.2, n_legit)

# Fraudulent transactions — different distribution
fraud = np.random.multivariate_normal(
    mean=[2,-2,1.5,-1.5,0.8,-0.8,1,-0.5],
    cov=np.eye(8)*0.5,
    size=n_fraud)
fraud_amount = np.random.lognormal(3.8, 0.8, n_fraud)

X_raw = np.vstack([legit, fraud])
amounts = np.concatenate([legit_amount, fraud_amount])
y_fraud = np.array([0]*n_legit + [1]*n_fraud)

# Scale amount (like the real dataset)
scaler_amt = StandardScaler()
X_full = np.column_stack([X_raw, scaler_amt.fit_transform(amounts.reshape(-1,1))])

print(f"Credit card transactions: {len(y_fraud):,}")
print(f"Legitimate: {n_legit:,}  ({n_legit/len(y_fraud)*100:.1f}%)")
print(f"Fraudulent: {n_fraud:,}  ({n_fraud/len(y_fraud)*100:.2f}%)")
print(f"\nClass imbalance ratio: {n_legit/n_fraud:.0f}:1")
print("\nThis is why accuracy is a TERRIBLE metric here:")
print(f"  Always predicting 'legit' gives {n_legit/len(y_fraud)*100:.2f}% accuracy")
print("  but catches exactly 0% of fraud!")

## 🎯 Part 1 — The Maximal Margin Classifier

SVM finds the hyperplane that **maximizes the margin** — the distance to the nearest training points (support vectors) from each class.

```
Maximize M  subject to:  yᵢ(β₀ + β₁xᵢ₁ + ... + βₚxᵢₚ) ≥ M
```

The **soft margin** (C parameter) allows some misclassification:
- Large C → narrow margin, few violations, may overfit
- Small C → wide margin, tolerates more violations, more robust

For fraud detection: we want high **recall on fraud** (catch as many as possible) even at the cost of some false alarms.

In [ ]:
# Use 2 most discriminating features for visualization
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_fraud, test_size=0.25,
                                             random_state=42, stratify=y_fraud)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Visualize decision boundary using top 2 features
X_2d = X_tr_s[:, :2]
y_2d = y_tr

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
C_vals   = [0.1, 1.0, 100.0]
titles   = ['Small C=0.1\n(wide margin, tolerant)',
            'C=1.0\n(balanced)',
            'Large C=100\n(tight margin, strict)']

for ax, C, title in zip(axes, C_vals, titles):
    svm = SVC(kernel='rbf', C=C, probability=True, class_weight='balanced')
    svm.fit(X_2d, y_2d)

    xx, yy = np.meshgrid(np.linspace(-4,4,200), np.linspace(-4,4,200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, colors=['#1e5fa8','#e85d20'])
    ax.scatter(X_2d[y_2d==0,0], X_2d[y_2d==0,1], color='#1e5fa8',
               alpha=0.3, s=8, label='Legit')
    ax.scatter(X_2d[y_2d==1,0], X_2d[y_2d==1,1], color='#e85d20',
               s=80, marker='x', lw=2, label='Fraud')

    fraud_recall = svm.score(X_2d[y_2d==1], y_2d[y_2d==1])
    ax.set_title(f'{title}\nFraud recall={fraud_recall:.0%}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('SVM Decision Boundaries — Credit Card Fraud\n'
             'Orange X marks = actual fraud transactions',
             fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## 🔮 Part 2 — The Kernel Trick

For nonlinear fraud patterns, the **RBF (Radial Basis Function) kernel** maps data to a higher-dimensional space where a linear boundary separates classes.

```
K(x,z) = exp(-γ||x-z||²)
```
γ controls how far a single training example's influence reaches — higher γ = tighter fit.

In [ ]:
# Full model with all features — RBF kernel, class_weight to handle imbalance
svm_full = SVC(kernel='rbf', C=1.0, gamma='scale',
               class_weight='balanced', probability=True, random_state=42)
svm_full.fit(X_tr_s, y_tr)
y_pred = svm_full.predict(X_te_s)
y_prob = svm_full.predict_proba(X_te_s)[:,1]

print("=== SVM on Credit Card Fraud ===\n")
print(classification_report(y_te, y_pred, target_names=['Legit','Fraud'],
                            digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Legit','Fraud']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\n(we care most about fraud recall)')

RocCurveDisplay.from_estimator(svm_full, X_te_s, y_te, ax=axes[1])
axes[1].set_title(f'ROC Curve (AUC={roc_auc_score(y_te,y_prob):.4f})')
plt.tight_layout(); plt.show()

n_sv = svm_full.n_support_.sum()
print(f"\n\u2714 Number of support vectors: {n_sv}")
print(f"   These are the only transactions that define the boundary")
print(f"   All other transactions can be removed without changing the model")

In [ ]:
# Business framing: threshold tuning
# In fraud detection, missing fraud (false negative) is far more costly than
# a false alarm (false positive). Adjust threshold to maximize fraud recall.
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_te, y_prob)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precision[:-1], color='#1e5fa8', lw=2, label='Precision')
ax.plot(thresholds, recall[:-1],    color='#e85d20', lw=2, label='Recall (fraud caught)')
ax.axvline(0.5, color='#888', lw=1.5, ls='--', label='Default threshold = 0.5')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision-Recall Tradeoff\n'
             'Lower threshold = catch more fraud but more false alarms')
ax.legend()
plt.tight_layout(); plt.show()

print("\n\u2714 Business decision: fraud cost >> false alarm cost")
print("   A threshold of 0.2-0.3 may be more appropriate than 0.5")
print("   This is a BUSINESS decision, not a modeling one")

In [ ]:
answers = {
    "q1": "",  # What are support vectors and why do only they define the boundary?
    "q2": "",  # What does the C parameter control and what happens at extremes?
    "q3": "",  # Why is accuracy a bad metric for fraud detection?
    "q4": "",  # What is the kernel trick and why does RBF work well for fraud?
    "q5": "",  # Should you lower the fraud threshold from 0.5? Justify your answer.
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Support Vector Machines"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*